# Evaluate Gemma 2B Fine-Tune: English → Twi Translation

This notebook evaluates your fine-tuned Gemma 2B model against the Khaya AI API
on 60 test sentences across 5 categories: general, fintech, healthcare, agriculture, voice_ui.

**Output:**
- BLEU scores per category
- Side-by-side comparison (Gemma vs Khaya)
- Recommendation: activate model, continue training, or use different base

**Runtime:** T4 GPU (free Colab tier is fine — inference only, no training)

**Before running:** Upload your model checkpoint to Google Drive or Hugging Face Hub
and set `MODEL_PATH` below.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
MODEL_PATH = "your-hf-username/gemma-2b-twi-finetuned"  # HF Hub repo OR Google Drive path
KHAYA_API_KEY = ""  # Your Khaya API key (from translation.ghananlp.org)
TEST_DATA_PATH = "data/test_sentences.json"  # Relative to this notebook
OUTPUT_PATH = "results/gemma_evaluation.json"

# Set to True to skip Khaya API calls (saves quota during debugging)
SKIP_KHAYA = False
# ──────────────────────────────────────────────────────────────────────────

In [ ]:
!pip install -q transformers accelerate bitsandbytes sacrebleu requests
import json, os, requests
from pathlib import Path
print("Packages installed.")

In [ ]:
# Load test set
with open(TEST_DATA_PATH) as f:
    data = json.load(f)

sentences = data["sentences"]
print(f"Loaded {len(sentences)} test sentences")
print("Categories:", {s['category'] for s in sentences})

In [ ]:
# ── Load Gemma 2B fine-tune ───────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print(f"Loading model from {MODEL_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    torch_dtype=torch.float16,
    load_in_4bit=True  # 4-bit quantization for T4 compatibility
)
print("Model loaded.")

In [ ]:
# ── Gemma inference ───────────────────────────────────────────────────────
def gemma_translate(text: str, max_new_tokens: int = 128) -> str:
    """Translate English to Twi using fine-tuned Gemma."""
    # Adjust this prompt to match whatever format you used during fine-tuning
    prompt = f"Translate English to Twi:\nEnglish: {text}\nTwi:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the Twi part after the prompt
    twi_part = decoded.split("Twi:")[-1].strip()
    return twi_part.split("\n")[0].strip()

# Quick smoke test
print(gemma_translate("Hello, how are you?"))

In [ ]:
# ── Khaya API ─────────────────────────────────────────────────────────────
def khaya_translate(text: str) -> str:
    """Translate English to Twi using Khaya AI API."""
    if SKIP_KHAYA:
        return "[skipped]"
    res = requests.post(
        "https://translation-api.ghananlp.org/v1/translate",
        headers={"Authorization": f"Bearer {KHAYA_API_KEY}", "Content-Type": "application/json"},
        json={"in": text, "lang": "en-tw"}
    )
    res.raise_for_status()
    return res.json()["translatedText"]

print(khaya_translate("Hello, how are you?"))

In [ ]:
# ── Run evaluation ────────────────────────────────────────────────────────
from tqdm import tqdm

results = []
for s in tqdm(sentences, desc="Evaluating"):
    gemma_out = gemma_translate(s["en"])
    khaya_out = khaya_translate(s["en"])
    results.append({
        "id": s["id"],
        "category": s["category"],
        "en": s["en"],
        "tw_ref": s["tw_ref"],
        "gemma": gemma_out,
        "khaya": khaya_out,
    })

print(f"Evaluated {len(results)} sentences")

In [ ]:
# ── BLEU scores ───────────────────────────────────────────────────────────
import sacrebleu
from collections import defaultdict

def bleu(hypotheses, references):
    return sacrebleu.corpus_bleu(hypotheses, [references]).score

categories = list({r["category"] for r in results})
print("\n── BLEU Scores ──────────────────────────────────────")
print(f"{'Category':<20} {'Gemma 2B':>10} {'Khaya API':>10}")
print("-" * 42)

summary = {}
for cat in sorted(categories):
    cat_results = [r for r in results if r["category"] == cat]
    refs  = [r["tw_ref"] for r in cat_results]
    gemma = [r["gemma"]  for r in cat_results]
    khaya = [r["khaya"]  for r in cat_results]
    gemma_bleu = bleu(gemma, refs)
    khaya_bleu = bleu(khaya, refs) if not SKIP_KHAYA else 0
    summary[cat] = {"gemma": gemma_bleu, "khaya": khaya_bleu}
    winner = "✓ Gemma" if gemma_bleu >= khaya_bleu else "✓ Khaya"
    print(f"{cat:<20} {gemma_bleu:>10.1f} {khaya_bleu:>10.1f}  {winner}")

all_refs  = [r["tw_ref"] for r in results]
all_gemma = [r["gemma"]  for r in results]
all_khaya = [r["khaya"]  for r in results]
print("-" * 42)
print(f"{'OVERALL':<20} {bleu(all_gemma, all_refs):>10.1f} {bleu(all_khaya, all_refs) if not SKIP_KHAYA else 0:>10.1f}")

In [ ]:
# ── Side-by-side sample ───────────────────────────────────────────────────
print("\n── Sample Outputs (first 5) ────────────────────────")
for r in results[:5]:
    print(f"\nEN:     {r['en']}")
    print(f"Ref:    {r['tw_ref']}")
    print(f"Gemma:  {r['gemma']}")
    print(f"Khaya:  {r['khaya']}")

In [ ]:
# ── Recommendation ────────────────────────────────────────────────────────
overall_gemma = bleu(all_gemma, all_refs)
overall_khaya = bleu(all_khaya, all_refs) if not SKIP_KHAYA else 0

print("\n── Recommendation ──────────────────────────────────")
if overall_gemma >= overall_khaya * 0.95:
    print("✅ ACTIVATE: Gemma 2B matches or exceeds Khaya quality.")
    print("   → Deploy model and set GEMMA_TW_URL in .env")
    print("   → Update registry.ts status from 'training' to 'active'")
elif overall_gemma >= overall_khaya * 0.80:
    print("⚠️  CLOSE: Gemma is within 20% of Khaya. Consider more training.")
    print("   → Add more domain-specific parallel pairs (fintech, healthcare)")
    print("   → Try fine-tuning on GhanaNLP Parallel Corpora (41K pairs)")
else:
    print("❌ NOT READY: Gemma needs significant improvement before replacing Khaya.")
    print("   → Check if fine-tuning prompt format matches inference prompt")
    print("   → Consider NLLB-200-distilled-600M as alternative base model")
    print("   → Run ml/03_collect_test_set.ipynb to review training data quality")

# Save results
Path("results").mkdir(exist_ok=True)
with open(OUTPUT_PATH, "w") as f:
    json.dump({"summary": summary, "results": results}, f, ensure_ascii=False, indent=2)
print(f"\nFull results saved to {OUTPUT_PATH}")